<a href="https://colab.research.google.com/github/Prahladh-Vulsa/Skill-RAG-Nexus/blob/main/Skill_RAG_Nexus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [ ]:
# Install all required packages
!pip install -qU langchain-google-genai
!pip install -qU langchain langchain-tavily
!pip install -qU langchain-mcp-adapters
!pip install -qU langchain-community pypdf
!pip install -qU langchain-text-splitter
!pip install -qU langchain-huggingface sentence_transformers
!pip install -qU langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.3.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.3.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
ERROR: Coul

Imports and Environment Setup

In [ ]:
import os
import requests
from google.colab import userdata
from sentence_transformers import CrossEncoder
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.agents import create_agent

In [ ]:
# 1. Load API keys cleanly
google_api_key = userdata.get('gemini_api_key')
tavily_api_key = userdata.get('tavily_apikey')
rapid_api_key = userdata.get('rapid_apikey')

# 2. Setup Gemini LLM
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

# 3. Setup Embedding and Reranker models
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 4. Text splitter configuration
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Simple PDF Ingestion

In [ ]:
def load_pdf_to_vectorstore(pdf_path, collection_name, persist_dir):
    """Loads a PDF if it exists, otherwise creates an empty collection cleanly."""
    store = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory=persist_dir
    )

    # Simple check: does the file exist?
    if os.path.exists(pdf_path):
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        splits = text_splitter.split_documents(docs)
        store.add_documents(splits)
        print(f"Loaded {len(splits)} chunks from {pdf_path}")
    else:
        print(f"Notice: '{pdf_path}' not found. Created empty store.")

    return store

# Process PDFs cleanly
research_vector_store = load_pdf_to_vectorstore(
    "/content/attention_is_all_you_need.pdf",
    "research_collection",
    "./chroma_research_db"
)

resume_vector_store = load_pdf_to_vectorstore(
    "/content/RESUME.pdf",
    "resume_collection",
    "./chroma_resume_db"
)

Loaded 52 chunks from /content/attention_is_all_you_need.pdf
Loaded 4 chunks from /content/RESUME.pdf


Custom Tools

In [ ]:
def rerank(query: str, docs: list, top_n: int = 3) -> list:
    """Helper to pick the top 3 best matching chunks using the Cross-Encoder."""
    if not docs:
        return []

    # Pair query with text, score them, and sort
    pairs = [[query, d.page_content] for d in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked[:top_n]]


@tool
def retrieve_from_research_pdf(query: str) -> str:
    """Retrieve precise academic facts from the research paper PDF."""
    docs = research_vector_store.similarity_search(query, k=10)
    top_docs = rerank(query, docs, top_n=3)

    if not top_docs:
        return "No relevant facts found in the research paper."

    return "\n\n".join([f"Content: {d.page_content}" for d in top_docs])


@tool
def retrieve_from_resume(query: str) -> str:
    """Retrieve skills and projects from the uploaded resume PDF."""
    docs = resume_vector_store.similarity_search(query, k=10)
    top_docs = rerank(query, docs, top_n=3)

    if not top_docs:
        return "No relevant details found in the resume."

    return "\n\n".join([f"Content: {d.page_content}" for d in top_docs])


@tool
def search_jobs(skill: str, location: str) -> str:
    """Search live job listings requiring specific skills via RapidAPI."""
    if not rapid_api_key:
        return "RapidAPI key is missing."

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {"x-rapidapi-key": rapid_api_key, "x-rapidapi-host": "jsearch.p.rapidapi.com"}
    params = {"query": f"{skill} in {location}", "page": "1", "country": "in"}

    response = requests.get(url, headers=headers, params=params)

    # Simple check on HTTP status
    if response.status_code != 200:
        return "Failed to fetch jobs from API."

    jobs = response.json().get("data", [])
    if not jobs:
        return f"No jobs found for {skill} in {location}."

    results = []
    for job in jobs[:3]:
        results.append(f"- {job.get('job_title')} at {job.get('employer_name')} ({job.get('job_apply_link')})")

    return "\n".join(results)

System Prompt

In [ ]:
system_prompt = """You are a multi-purpose assistant capable of executing technical research, career counseling, and job matching.

You have access to these specific tools:
1. mcp_tavily: Search web trends or broad updates.
2. search_jobs: Search live job listings based on skills and locations.
3. retrieve_from_research_pdf: Query facts from the research paper.
4. retrieve_from_resume: Query skills and experience from the resume.

Routing Strategy:
- Use retrieve_from_research_pdf for paper/concept questions.
- Use retrieve_from_resume to extract candidate skills first, then use search_jobs to find openings.
- Present job results cleanly in plain text with application links.
"""

Agent Runner

In [ ]:
async def run_unified_agent(user_query: str):
    # Fetch MCP Tavily tools dynamically
    client = MultiServerMCPClient({
        "mcp_tavily": {
            "transport": "http",
            "url": f"https://mcp.tavily.com/mcp/?tavilyApiKey={tavily_api_key}",
        }
    })
    mcp_tools = await client.get_tools()

    # Combine all tools
    all_tools = mcp_tools + [search_jobs, retrieve_from_research_pdf, retrieve_from_resume]

    # Create and execute the agent
    agent = create_agent(
        model=model,
        tools=all_tools,
        system_prompt=system_prompt,
        debug=True
    )

    response = await agent.ainvoke({
        "messages": [{"role": "user", "content": user_query}]
    })

    print("\n--- AGENT RESPONSE ---")
    print(response["messages"][-1].content[0]["text"])

Run Query

In [ ]:
query = """
1. What are the main concepts explained in the research paper?
2. Check my resume to see if I have experience related to those concepts.
3. Find 2 matching live job openings in Bangalore.
"""

await run_unified_agent(query)

[values] {'messages': [HumanMessage(content='\n1. What are the main concepts explained in the research paper?\n2. Check my resume to see if I have experience related to those concepts.\n3. Find 2 matching live job openings in Bangalore.\n', additional_kwargs={}, response_metadata={}, id='4eb3b6a1-060f-458d-b1e9-2dd88813ed3f')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'retrieve_from_research_pdf', 'arguments': '{"query": "main concepts explained in the research paper"}'}, '__gemini_function_call_thought_signatures__': {'69721c67-6fc6-4d67-9f5c-d3bfa22cfafa': 'CtsFARFNMg/3DLoKIjua7Db8xNdvPJwAqWtMZb3ipX0KRt2waCVtiy7GaX9xrcLX0TcMGwns8jlRuygHZiOelqB7X/pJV7WurA5tlAF92lAaE5zJz8bD/0wTSbsCZFvNQG710oKuUtaMFPLfZlDN3oj5Gc8qVpZn4z5Yt9M/Fd51OwCdL8xry9g2Sl4E5wqOf6/C1dXJXteYUfW0iGIq3uLmboGpDk3vubHof8bpYnVvBT2BfAjBh/B+kLc0bIII3LgjRYfu/ATKCIJQxvaR/GVMaPcfnSkJi/kP+4zadJdUe7/Y8Hx0tbNk3ip0NdWgKA0/PG9V5MiRdHkbW/SvqHrUJAftyJsUDZMJuZzI4q00yXwWa1SAGd

[updates] {'tools': {'messages': [ToolMessage(content='Content: [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated\ncorpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1993.\n[26] David McClosky, Eugene Charniak, and Mark Johnson. Effective self-training for parsing. In\nProceedings of the Human Language Technology Conference of the NAACL, Main Conference ,\npages 152–159. ACL, June 2006.\n[27] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention\nmodel. In Empirical Methods in Natural Language Processing , 2016.\n[28] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive\nsummarization. arXiv preprint arXiv:1705.04304, 2017.\n[29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Klein. Learning accurate, compact,\nand interpretable tree annotation. In Proceedings of the 21st International Conference on\nComputational Linguistics and 44

[updates] {'tools': {'messages': [ToolMessage(content='Content: 2021 – 2025\nJun 2024 – Aug 2024\nVULSA PRAHLADH+91 8374371435 |  prahladvulsa30@gmail.com |  linkedin.com/in/prahladh-vulsa |  github.com/Prahladh-Vulsa \nOBJECTIVE\nAspiring AI Engineer with hands-on experience in Generative AI, LLM integration, and workflow automation. Passionate\nabout building AI-powered applications that improve efficiency and solve real-world problems. Eager to leverage modern\nAI technologies to develop scalable, impactful solutions that deliver business value. \nEDUCATION\nSathyabama University — B.E. Computer Science, Cyber Security (CGPA: 7.67)\nSKILLS\nAI & ML: Model Context Protocol (MCP), AI Agents, Retrieval-Augmented Generation (RAG),\nGenerative AI, Large Language Models (Gemini, Mistral), Prompt Engineering, Agentic\nWorkflows & Session Memory, Hugging Face\nFrameworks & Tools:LangChain, langchain-mcp-adapters, MultiServerMCPClient, LangGraph / Checkpointers\n(InMemorySaver), Streamlit, G

[updates] {'tools': {'messages': [ToolMessage(content='- Generative AI Solution Architect at Infosys (https://apna.co/job/bengaluru/gen-ai-architect-1143589072)\n- Generative AI Engineer / GenAI Engineer/AI Engineer/Machine Learning E (Bengaluru) at Wroots Global (https://in.jobrapido.com/jobpreview/3888933541093834752)\n- Gen AI/LLM Engineer_5_ years at Zorba AI (https://in.linkedin.com/jobs/view/gen-ai-llm-engineer-5-years-at-zorba-ai-4438328941)', name='search_jobs', id='c489df7c-3b0b-46cd-8e36-479c3362f111', tool_call_id='0740ad98-50d3-420e-a707-662f25dbec53')]}}
[values] {'messages': [HumanMessage(content='\n1. What are the main concepts explained in the research paper?\n2. Check my resume to see if I have experience related to those concepts.\n3. Find 2 matching live job openings in Bangalore.\n', additional_kwargs={}, response_metadata={}, id='4eb3b6a1-060f-458d-b1e9-2dd88813ed3f'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'retrieve_from_research_pdf', 